# Phase 4 Evaluation Report: Trust-Weighted Multi-Agent Co-Learning
This notebook evaluates the performance differences between the Phase 2 baseline (S1) and the Phase 3 trust-weighted imitation policies (S2).

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set_theme(style="whitegrid")

ModuleNotFoundError: No module named 'seaborn'

## Data Loading
We load the baseline aggregates and the parameter sweeps from `outputs/ablations`.

In [ ]:
root = Path("outputs/ablations")
runs = []
for file in root.glob("**/runs/*_summary.parquet"):
    run_df = pd.read_parquet(file)
    run_df["experiment_name"] = file.parts[-3] # ablation_root / run_name / runs / file
    runs.append(run_df)

if runs:
    df = pd.concat(runs, ignore_index=True)
    baseline = df[df["experiment_name"] == "phase2_baseline"]
    ablations = df[df["experiment_name"] != "phase2_baseline"]
else:
    print("No outputs found in outputs/ablations!")
    df = pd.DataFrame()

## RQ1: Trust Convergence
We visualize trust entropy and asymmetry tracking from the baseline.

In [ ]:
def plot_running_metrics(train_file):
    steps = pd.read_parquet(train_file)
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    
    # Entropy over steps
    if 'entropy_0' in steps.columns:
        for i in range(5):
            axes[0].plot(steps["step"], steps[f"entropy_{i}"], label=f"Agent {i}")
    axes[0].set_title("Trust Entropy over Time")
    axes[0].set_xlabel("Steps")
    
    # Asymmetry
    if 'asymmetry_index' in steps.columns:
        axes[1].plot(steps["step"], steps["asymmetry_index"], color='purple')
    axes[1].set_title("Asymmetry Index over Time")
    axes[1].set_xlabel("Steps")
    
    plt.tight_layout()
    plt.show()

sample_file = root / "phase2_baseline" / "train" / "steps" / "seed_02026_steps.parquet"
if sample_file.exists():
    plot_running_metrics(sample_file)

## RQ2: Performance Improvement (S1 vs S2)
Using final Sharpe Ratio means as the primary comparison metric.

In [ ]:
if not df.empty:
    eval_sharpe_cols = [c for c in df.columns if 'eval_final_sharpe_mean_episode' in c]

    if len(eval_sharpe_cols) > 0:
        plot_data = []
        
        for _, row in baseline.iterrows():
            plot_data.append({"Experiment": "Baseline (S1)", "Sharpe": row[eval_sharpe_cols[-1]], "Beta": 0.0, "Predictor": "None"})
            
        for _, row in ablations.iterrows():
            name = row["experiment_name"]
            beta = name.split("_beta_")[1].replace("p", ".") if "_beta_" in name else "0.0"
            pred = name.split("_beta_")[0].replace("pred_", "") if "pred_" in name else "unknown"
            plot_data.append({"Experiment": f"{pred} (B={beta})", "Sharpe": row[eval_sharpe_cols[-1]], "Beta": float(beta), "Predictor": pred})
            
        plot_df = pd.DataFrame(plot_data)
        
        plt.figure(figsize=(14, 6))
        sns.boxplot(data=plot_df, x="Experiment", y="Sharpe")
        plt.xticks(rotation=45, ha="right")
        plt.title("Distribution of Final Evaluation Sharpe Ratios")
        plt.tight_layout()
        plt.show()

## RQ3: Asymmetric Trust Analysis
Does trust magnitude correlate with increased sharpe?

In [ ]:
if not df.empty and 'final_eval_asymmetry' in df.columns and len(eval_sharpe_cols) > 0:
    plt.figure(figsize=(6, 5))
    sns.scatterplot(data=df, x="final_eval_asymmetry", y=eval_sharpe_cols[-1], hue="experiment_name", legend=False)
    plt.title("Asymmetry vs Sharpe Ratio across seeds/ablations")
    plt.xlabel("Final Eval Asymmetry")
    plt.ylabel("Final Eval Sharpe Ratio")
    plt.grid(True)
    plt.show()

## Ablation Sweeps
Effect of `imitation_beta` on the Sharpe ratio.

In [ ]:
if not df.empty and len(eval_sharpe_cols) > 0:
    s2_data = plot_df[plot_df["Experiment"] != "Baseline (S1)"]
    if not s2_data.empty:
        plt.figure(figsize=(8, 5))
        sns.lineplot(data=s2_data, x="Beta", y="Sharpe", hue="Predictor", marker="o", err_style="bars")
        plt.title("Sensitivity to Imitation Beta")
        plt.axhline(plot_df[plot_df["Experiment"] == "Baseline (S1)"]["Sharpe"].mean(), color='r', linestyle='--', label="Baseline Mean")
        plt.legend()
        plt.grid(True)
        plt.show()